# Prepare Significant Proteomics (Canonical Workflow)

This notebook reproduces the production significance filtering workflow used for Figure 6.

It performs per-compound vs reference gene filtering and saves:
- `Data/Base/X_proteomics_sig.pkl`
- `Data/Special/X_proteomics_sig.pkl`

Notes:
- Inputs are resolved from `DATA_ROOT` and require `Base`, `Special`, `Fasta_files`, and `MS_reference_files`.
- `X_samples` is built as `concat(Base/X_proteomics.pkl, Special/X_proteomics.pkl)`.


In [2]:
from pathlib import Path
import os
import re
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from Bio import SeqIO


# -------- Resolve DATA_ROOT --------
cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents]

candidates = [root / "Data" for root in search_roots]
req_dirs = ["MS_reference_files", "Base", "Special", "Fasta_files"]

DATA_ROOT = None
for cand in candidates:
    if cand.exists() and all((cand / d).exists() for d in req_dirs):
        DATA_ROOT = cand
        break

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not resolve DATA_ROOT with required subdirs: "
        f"{req_dirs}. Checked: {[str(c) for c in candidates]}"
    )

FASTA_PATH = DATA_ROOT / "Fasta_files" / "Ecoli.fasta"
REFERENCE_DIR = DATA_ROOT / "MS_reference_files"
BASE_DIR = DATA_ROOT / "Base"
SPECIAL_DIR = DATA_ROOT / "Special"

print("DATA_ROOT:", DATA_ROOT)
print("Reference dir:", REFERENCE_DIR)
print("Base proteomics:", BASE_DIR / "X_proteomics.pkl")
print("Special proteomics:", SPECIAL_DIR / "X_proteomics.pkl")


DATA_ROOT: /mnt/nas/PhD/Code/Ms_paint/code_repository/Data
Reference dir: /mnt/nas/PhD/Code/Ms_paint/code_repository/Data/MS_reference_files
Base proteomics: /mnt/nas/PhD/Code/Ms_paint/code_repository/Data/Base/X_proteomics.pkl
Special proteomics: /mnt/nas/PhD/Code/Ms_paint/code_repository/Data/Special/X_proteomics.pkl


In [9]:
# -------- Minimal preprocessing helpers (from canonical workflow) --------

def parse_uniprot_fasta_to_df(fasta_file: str | Path) -> pd.DataFrame:
    records = []
    for record in SeqIO.parse(str(fasta_file), "fasta"):
        description = record.description
        gene_name = re.search(r'GN=([\w\d\-]+)', description)
        gene_name = gene_name.group(1) if gene_name else None
        records.append({"Gene_Name": gene_name})
    return pd.DataFrame(records)


def is_intensity_column(col_name: str) -> bool:
    excluded_prefixes = ("C: ", "N: ", "T: ")
    return not col_name.startswith(excluded_prefixes)


def filter_intensity_and_metadata(df: pd.DataFrame, metadata_cols=None) -> pd.DataFrame:
    if metadata_cols is None:
        metadata_cols = ["T: Genes"]
    intensity_cols = [c for c in df.columns if is_intensity_column(c)]
    selected_cols = intensity_cols + metadata_cols
    return df[selected_cols]


def expand_gene_rows(df: pd.DataFrame, gene_col: str = "T: Genes") -> pd.DataFrame:
    expanded_rows = []
    for _, row in df.iterrows():
        genes = str(row[gene_col]).split(';')
        for gene in genes:
            new_row = row.copy()
            new_row[gene_col] = gene.strip()
            expanded_rows.append(new_row)
    return pd.DataFrame(expanded_rows)


def create_combined_fingerprint(filtered_dataframes, fasta_df: pd.DataFrame) -> pd.DataFrame:
    all_genes = sorted(fasta_df["Gene_Name"].dropna().unique())
    combined_df = pd.DataFrame(index=all_genes)

    for df in filtered_dataframes:
        df = df.rename(columns={"T: Genes": "Gene_Name"})
        df = expand_gene_rows(df, gene_col="Gene_Name")
        replicate_cols = [c for c in df.columns if c not in ["Gene_Name"]]

        for col in replicate_cols:
            temp_df = df[["Gene_Name", col]].copy().set_index("Gene_Name")
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=pd.errors.PerformanceWarning)
                combined_df[col] = temp_df[col]

    return combined_df.fillna(0)


def preprocess_X(df: pd.DataFrame) -> pd.DataFrame:
    return df.replace(',', '.', regex=True).apply(pd.to_numeric, errors='coerce')


def safe_simplify_reference_names(index_like) -> pd.Index:
    out = []
    for name in map(str, index_like):
        n = name.strip()
        parts = n.split("_")
        jh_parts = [p for p in parts if re.fullmatch(r'JH\d+', p)]
        if jh_parts and len(parts) >= 2:
            out.append(f"{jh_parts[-1]}_{parts[-1]}")
        elif len(parts) >= 2:
            out.append(f"{parts[-2]}_{parts[-1]}")
        else:
            out.append(n)
    return pd.Index(out)


def clean_reference_names(index_like: pd.Index) -> pd.Index:
    cleaned = []
    for raw in map(str, index_like):
        n = raw.strip()

        m = re.match(r'^(DA)(\d+)_G[A-Z]\d+$', n)
        if m:
            cleaned.append(f"{m.group(1)}_{m.group(2)}")
            continue

        m = re.match(r'^(DB)(\d+)_G[A-Z]\d+$', n)
        if m:
            cleaned.append(f"{m.group(1)}_{m.group(2)}")
            continue

        m = re.match(r'^Ctrl[_\s]?10[_\-]?(\d+)$', n)
        if m:
            cleaned.append(f"Ctrl_10_{m.group(1)}")
            continue

        cleaned.append(n)
    return pd.Index(cleaned)


def is_data_file(filename: str) -> bool:
    return filename.lower().endswith((".txt", ".tsv")) and not filename.lower().endswith(".fasta")


def _auto_gene_col(df: pd.DataFrame, default: str = "T: Genes") -> str:
    if default in df.columns:
        return default
    candidates = [c for c in df.columns if c.strip().lower() in {
        "t: genes", "genes", "gene", "gene names", "majority protein ids"
    }]
    if not candidates:
        raise KeyError(f"Could not find '{default}' or a gene column in reference table.")
    return candidates[0]


def prepare_reference_matrix(fasta_file: str | Path, ref_folder: str | Path) -> pd.DataFrame:
    fasta_df = parse_uniprot_fasta_to_df(fasta_file)

    filtered_tables = []
    ref_folder = Path(ref_folder)
    for fname in os.listdir(ref_folder):
        fpath = ref_folder / fname
        if not fpath.is_file() or not is_data_file(fname):
            continue

        df = pd.read_csv(fpath, sep="	", dtype=str)
        gene_col = _auto_gene_col(df)
        df_keep = filter_intensity_and_metadata(df, metadata_cols=[gene_col])

        df_keep = df_keep.rename(columns={gene_col: "T: Genes"})
        df_expanded = expand_gene_rows(df_keep, gene_col="T: Genes")

        meas_cols = [c for c in df_expanded.columns if c != "T: Genes"]
        df_expanded[meas_cols] = (
            df_expanded[meas_cols]
            .replace(',', '.', regex=True)
            .apply(pd.to_numeric, errors='coerce')
        )

        filtered_tables.append(df_expanded)

    if not filtered_tables:
        raise FileNotFoundError(f"No reference .txt/.tsv files found in {ref_folder}")

    combined = create_combined_fingerprint(filtered_tables, fasta_df)
    X_ref = combined.T.copy()
    X_ref.index = safe_simplify_reference_names(pd.Index(X_ref.index))
    X_ref = preprocess_X(X_ref).fillna(0.0)

    if X_ref.shape[1] != 4401:
        warnings.warn(f"prepare_reference_matrix: expected 4401 features, got {X_ref.shape[1]}.")

    return X_ref


In [10]:
# -------- Load canonical inputs --------
X_base = pd.read_pickle(BASE_DIR / "X_proteomics.pkl")
X_special = pd.read_pickle(SPECIAL_DIR / "X_proteomics.pkl")
X_samples = pd.concat([X_base, X_special], axis=0)

if X_samples.index.duplicated().any():
    dup_n = int(X_samples.index.duplicated().sum())
    raise ValueError(f"Duplicate sample IDs after concat(Base, Special): {dup_n}")

X_ref = prepare_reference_matrix(FASTA_PATH, REFERENCE_DIR)
X_ref.index = clean_reference_names(X_ref.index)

if X_ref.index.duplicated().any():
    X_ref = X_ref.groupby(X_ref.index).mean(numeric_only=True).sort_index()

print("X_base:", X_base.shape)
print("X_special:", X_special.shape)
print("X_samples (concat):", X_samples.shape)
print("X_ref:", X_ref.shape)
print("X_ref unique rows:", len(X_ref.index))


X_base: (200, 4401)
X_special: (20, 4401)
X_samples (concat): (220, 4401)
X_ref: (71, 4401)
X_ref unique rows: 71


In [11]:
# -------- Compound -> control mapping (canonical) --------
group_to_jhnums = {
    "DM": [68, 67, 66, 43],
    "DL": [65, 64, 53, 51],
    "DK": [62, 61, 58, 21, 20],
    "DI": [12, 3, 0],
    "10": [10],
    "DH": [18, 15, 14, 8, 6],
    "HH": [44],
    "DG": [47, 46, 45],
    "DF": [39, 37, 33, 32, 31],
    "HC": [29],
    "DE": [26, 9, 7],
    "DD": [16, 5, 2, 23],
    "DC": [38, 30, 24],
    "DB": [56, 50, 49, 48, 40, 42],
    "DA": [41, 28],
    "H": [27],
    "Punkt_D": [36, 35],
    "D": [13, 17, 55],
}


def refs_for_group(group: str, ref_index: pd.Index) -> list[str]:
    g = re.escape(group)
    pat_underscore = re.compile(rf"^{g}_[0-9]+$")
    pat_nounders = re.compile(rf"^{g}[0-9]+$")
    hits = [name for name in ref_index if pat_underscore.match(str(name)) or pat_nounders.match(str(name))]
    return sorted(hits)


group_to_refs = {g: refs_for_group(g, X_ref.index) for g in group_to_jhnums.keys()}
for g, refs in group_to_refs.items():
    if not refs:
        print(f"[WARN] No refs found for group {g}")


## Section 1: Canonical Significance Filtering (used in production)

This section reproduces the original significance logic:
- test each gene for each compound vs its mapped reference group,
- ignore zeros as missing,
- keep significant genes, set non-significant genes to zero,
- save split-aligned outputs to `Data/Base` and `Data/Special` as `X_proteomics_sig.pkl`.


In [12]:
# -------- Canonical significance test (same as original workflow) --------

def ttest_equalvar_ignore_zeros(comp_vals, ref_vals, alpha=0.05):
    comp = np.array(comp_vals, dtype=float)
    ref = np.array(ref_vals, dtype=float)

    comp = comp[comp != 0]
    ref = ref[ref != 0]

    n_c = comp.size
    n_r = ref.size

    if n_r < 2:
        return False, 1.0

    if n_c >= 2:
        _t, p = stats.ttest_ind(comp, ref, equal_var=True, alternative="two-sided")
        if np.isnan(p):
            return False, 1.0
        return p <= alpha, float(p)

    if n_c == 1 and n_r >= 3:
        x = comp[0]
        m_r = np.mean(ref)
        s_r = np.std(ref, ddof=1)
        if s_r <= 0:
            return False, 1.0
        t = (x - m_r) / (s_r * np.sqrt(1.0 + 1.0 / n_r))
        df = n_r - 1
        p = 2.0 * stats.t.sf(np.abs(t), df)
        return p <= alpha, float(p)

    return False, 1.0


def parse_jh_number(sample_name: str):
    m = re.match(r"^JH(\d+)_", str(sample_name))
    return int(m.group(1)) if m else None


jh_to_group = {n: g for g, nums in group_to_jhnums.items() for n in nums}

compound_sets = {}
for s in X_samples.index:
    jh = parse_jh_number(s)
    if jh is None:
        continue
    compound_sets.setdefault(f"JH{jh}", []).append(s)

X_filtered_sig = X_samples.copy()

for comp_key, comp_rows in sorted(compound_sets.items(), key=lambda kv: int(kv[0][2:])):
    jh = int(comp_key[2:])
    ctrl_group = jh_to_group.get(jh)
    if ctrl_group is None:
        continue
    ref_rows = group_to_refs.get(ctrl_group, [])
    if not ref_rows:
        continue

    Xc = X_samples.loc[comp_rows]
    Xr = X_ref.loc[ref_rows]
    genes = Xc.columns.intersection(Xr.columns)
    Xc = Xc[genes].copy()
    Xr = Xr[genes].copy()

    sig_by_gene = pd.Series(False, index=genes)
    for g in genes:
        is_sig, _p = ttest_equalvar_ignore_zeros(Xc[g].values, Xr[g].values, alpha=0.05)
        sig_by_gene[g] = is_sig

    sig_cols = sig_by_gene.index[sig_by_gene.values]
    nonsig_cols = sig_by_gene.index[~sig_by_gene.values]

    X_filtered_sig.loc[comp_rows, sig_cols] = X_samples.loc[comp_rows, sig_cols]
    X_filtered_sig.loc[comp_rows, nonsig_cols] = 0.0

changed = (X_filtered_sig.values != X_samples.values)
print("Canonical filtering complete")
print("Changed cells:", int(changed.sum()), "of", changed.size)


Canonical filtering complete
Changed cells: 418613 of 968220


In [ ]:
# -------- Save canonical outputs exactly as used --------

def align_columns_or_raise(df_target: pd.DataFrame, df_reference: pd.DataFrame, name: str):
    missing = df_reference.columns.difference(df_target.columns)
    extra = df_target.columns.difference(df_reference.columns)
    if len(missing) or len(extra):
        msg = []
        if len(missing):
            msg.append(f"{name}: missing {len(missing)} columns")
        if len(extra):
            msg.append(f"{name}: unexpected {len(extra)} columns")
        raise ValueError("Column mismatch:\n" + "\n".join(msg))
    return df_target.loc[:, df_reference.columns]

X_base = pd.read_pickle(BASE_DIR / "X_proteomics.pkl")
X_special = pd.read_pickle(SPECIAL_DIR / "X_proteomics.pkl")

X_filtered_for_base = align_columns_or_raise(X_filtered_sig, X_base, name="[Base]")
X_filtered_for_special = align_columns_or_raise(X_filtered_sig, X_special, name="[Special]")

X_base_sig = X_filtered_for_base.reindex(X_base.index).fillna(0.0)
X_special_sig = X_filtered_for_special.reindex(X_special.index).fillna(0.0)

assert X_base_sig.columns.equals(X_base.columns)
assert X_special_sig.columns.equals(X_special.columns)

out_base = BASE_DIR / "X_proteomics_sig.pkl"
out_special = SPECIAL_DIR / "X_proteomics_sig.pkl"

X_base_sig.to_pickle(out_base)
X_special_sig.to_pickle(out_special)


print("Saved canonical outputs:")
print(" -", out_base, X_base_sig.shape)
print(" -", out_special, X_special_sig.shape)


Saved canonical outputs:
 - /mnt/nas/PhD/Code/Ms_paint/code_repository/Data/Base/X_proteomics_sig.pkl (200, 4401)
 - /mnt/nas/PhD/Code/Ms_paint/code_repository/Data/Special/X_proteomics_sig.pkl (20, 4401)
